# Visibility-cut breakdown: `gp_in_tracker` vs `gp_in_calo`

**Confirmed against the actual source of `ttbar_reco_10000.parquet`** (`postprocessing.py`,
`--dataset muoncollider` branch). The mask that decides what enters `targetjet` is exactly:

```python
gp_in_tracker = gp_to_track >= 0.2                      # TRACKER_WEIGHT_THRESHOLD
gp_in_calo    = gp_to_cluster / energy > 0.05            # CALO_ENERGY_FRACTION_THRESHOLD
mask_visible  = (generatorStatus == 1) & (gp_in_tracker | gp_in_calo)
```

`genjet` (truth) is built from *all* status-1 particles, no cut. `targetjet` is built only
from particles surviving `mask_visible` -- i.e. this cut is a pre-filter applied *before*
jet clustering, not an energy adjustment applied after.

Every status-1 particle falls into exactly one of four mutually exclusive, exhaustive
buckets: track-only, calo-only, both, or neither (= dropped from `targetjet`).


In [ ]:
# Thresholds exactly as in postprocessing.py -- named here for readability, not hardcoded inline.
TRACKER_WEIGHT_THRESHOLD = 0.2          # gp_to_track >= this -> gp_in_tracker
CALO_ENERGY_FRACTION_THRESHOLD = 0.05   # gp_to_cluster / energy > this -> gp_in_calo

from scipy.sparse import coo_matrix

link_data = ev.arrays([
    "CalohitMCTruthLink.weight",
    "_CalohitMCTruthLink_from/_CalohitMCTruthLink_from.collectionID",
    "_CalohitMCTruthLink_from/_CalohitMCTruthLink_from.index",
    "_CalohitMCTruthLink_to/_CalohitMCTruthLink_to.collectionID",
    "_CalohitMCTruthLink_to/_CalohitMCTruthLink_to.index",
    "SiTracksMCTruthLink.weight",
    "_SiTracksMCTruthLink_from/_SiTracksMCTruthLink_from.index",
    "_SiTracksMCTruthLink_to/_SiTracksMCTruthLink_to.index",
    "_PandoraClusters_hits/_PandoraClusters_hits.collectionID",
    "_PandoraClusters_hits/_PandoraClusters_hits.index",
    "PandoraClusters/PandoraClusters.hits_begin",
    "PandoraClusters/PandoraClusters.hits_end",
    "SiTracks_Refitted/SiTracks_Refitted.type",
])

mc_full = ev.arrays([
    "MCParticle/MCParticle.PDG",
    "MCParticle/MCParticle.generatorStatus",
    "MCParticle/MCParticle.simulatorStatus",
    "MCParticle/MCParticle.charge",
    "MCParticle/MCParticle.mass",
    "MCParticle/MCParticle.momentum.x",
    "MCParticle/MCParticle.momentum.y",
    "MCParticle/MCParticle.momentum.z",
])


def analyze_event_visibility(iev):
    """Per-genparticle: PDG, generatorStatus, simulatorStatus, energy, eta, gp_in_tracker,
    gp_in_calo, px, py, pz, mass, and the RAW gp_to_track / gp_to_cluster values (not just
    the boolean cut) -- kept so later cells can look at *how close* a dropped particle was
    to the threshold, not just whether it passed. Full MCParticle index space, not
    pre-filtered to status==1.
    """

    # --- global hit index, built the same way postprocessing.py's get_calohit_matrix_and_genadj does ---
    hit_idx_global = 0
    hit_idx_local_to_global = {}
    for col in sorted(hit_data.keys()):
        icol = collectionIDs[col]
        n_hits_this_coll = len(hit_data[col][col + ".cellID"][iev])
        for ihit in range(n_hits_this_coll):
            hit_idx_local_to_global[(icol, ihit)] = hit_idx_global
            hit_idx_global += 1
    n_hit = hit_idx_global

    # CalohitMCTruthLink points at the *Digi*-level calo collections
    # (EcalBarrelCollectionDigi, EcalEndcapCollectionDigi, HcalBarrelCollectionDigi,
    # HcalEndcapCollectionDigi), not the *Rec* collections hit_data/hit_idx_local_to_global
    # are built from -- so those collectionIDs are never keys in the map above, and a
    # naive lookup KeyErrors. Confirmed directly against this file that for every event,
    # each Digi collection's cellID array is identical, same-order, to its Rec
    # counterpart's -- i.e. Digi hit index i == Rec hit index i -- so we just remap the
    # Digi collectionID to the corresponding Rec collectionID before indexing.
    # (postprocessing.py does the same remap for the real target construction, so this
    # is not a source of bias -- it was only missing from this notebook's own
    # reimplementation.)
    digi_to_rec_colname = {
        "EcalBarrelCollectionDigi": "EcalBarrelCollectionRec",
        "EcalEndcapCollectionDigi": "EcalEndcapCollectionRec",
        "HcalBarrelCollectionDigi": "HcalBarrelCollectionRec",
        "HcalEndcapCollectionDigi": "HcalEndcapCollectionRec",
    }
    digi_colid_to_rec_colid = {
        collectionIDs[digi]: collectionIDs[rec] for digi, rec in digi_to_rec_colname.items()
    }

    # --- genparticle <-> calohit adjacency ---
    calo_from_colid = link_data["_CalohitMCTruthLink_from/_CalohitMCTruthLink_from.collectionID"][iev]
    calo_from_idx = link_data["_CalohitMCTruthLink_from/_CalohitMCTruthLink_from.index"][iev]
    calo_to_idx = link_data["_CalohitMCTruthLink_to/_CalohitMCTruthLink_to.index"][iev]
    calo_w = link_data["CalohitMCTruthLink.weight"][iev]

    gp_hit_gp, gp_hit_hit, gp_hit_w = [], [], []
    for colid, idx, gen_idx, w in zip(calo_from_colid, calo_from_idx, calo_to_idx, calo_w):
        rec_colid = digi_colid_to_rec_colid.get(colid, colid)
        gp_hit_gp.append(gen_idx)
        gp_hit_hit.append(hit_idx_local_to_global[(rec_colid, idx)])
        gp_hit_w.append(w)

    # --- hit <-> cluster adjacency ---
    cl_coll = link_data["_PandoraClusters_hits/_PandoraClusters_hits.collectionID"][iev]
    cl_idx = link_data["_PandoraClusters_hits/_PandoraClusters_hits.index"][iev]
    hits_begin = link_data["PandoraClusters/PandoraClusters.hits_begin"][iev]
    hits_end = link_data["PandoraClusters/PandoraClusters.hits_end"][iev]

    hit_cl_hit, hit_cl_cl = [], []
    for icluster in range(len(hits_begin)):
        hbeg, hend = hits_begin[icluster], hits_end[icluster]
        for icol, idx in zip(cl_coll[hbeg:hend], cl_idx[hbeg:hend]):
            hit_cl_hit.append(hit_idx_local_to_global[(icol, idx)])
            hit_cl_cl.append(icluster)
    n_cluster = len(hits_begin)

    # --- genparticle <-> track adjacency ---
    trk_gen_trkidx = link_data["_SiTracksMCTruthLink_from/_SiTracksMCTruthLink_from.index"][iev]
    trk_gen_genidx = link_data["_SiTracksMCTruthLink_to/_SiTracksMCTruthLink_to.index"][iev]
    trk_gen_w = link_data["SiTracksMCTruthLink.weight"][iev]
    n_track = len(link_data["SiTracks_Refitted/SiTracks_Refitted.type"][iev])

    pdg = mc_full["MCParticle/MCParticle.PDG"][iev]
    genstatus = mc_full["MCParticle/MCParticle.generatorStatus"][iev]
    simstatus = mc_full["MCParticle/MCParticle.simulatorStatus"][iev]
    mass = mc_full["MCParticle/MCParticle.mass"][iev]
    px = mc_full["MCParticle/MCParticle.momentum.x"][iev]
    py = mc_full["MCParticle/MCParticle.momentum.y"][iev]
    pz = mc_full["MCParticle/MCParticle.momentum.z"][iev]
    n_gp = len(pdg)

    # vectorized energy/eta (event 4 alone has ~700k MCParticles -- a per-particle
    # vector.obj() python loop takes minutes there; use vector.awk like
    # postprocessing.py's gen_to_features does, which is fast at any n_gp)
    p4 = vector.awk(awkward.zip({
        "mass": awkward.Array(np.asarray(mass)),
        "x": awkward.Array(np.asarray(px)),
        "y": awkward.Array(np.asarray(py)),
        "z": awkward.Array(np.asarray(pz)),
    }))
    energy = np.asarray(p4.energy)
    eta = np.asarray(p4.eta)

    # gp_to_track: max weight (hit-count fraction) across matched tracks, per genparticle
    if len(trk_gen_genidx) > 0:
        gp_to_track = np.asarray(
            coo_matrix((np.asarray(trk_gen_w), (np.asarray(trk_gen_genidx), np.asarray(trk_gen_trkidx))),
                       shape=(n_gp, max(n_track, 1))).max(axis=1).todense()
        )[:, 0]
    else:
        gp_to_track = np.zeros(n_gp)

    # gp_to_cluster: total calo-hit link-weight attributed to this genparticle via any cluster
    # NOTE: this is NOT calibrated to GeV -- see the units check later in this notebook.
    gp_to_calohit = coo_matrix((gp_hit_w, (gp_hit_gp, gp_hit_hit)), shape=(n_gp, max(n_hit, 1)))
    calohit_to_cluster = coo_matrix((np.ones(len(hit_cl_hit)), (hit_cl_hit, hit_cl_cl)), shape=(n_hit, max(n_cluster, 1)))
    gp_to_cluster = np.asarray((gp_to_calohit * calohit_to_cluster).sum(axis=1))[:, 0]

    gp_in_tracker = gp_to_track >= TRACKER_WEIGHT_THRESHOLD
    with np.errstate(divide="ignore", invalid="ignore"):
        gp_in_calo = np.where(energy > 0, gp_to_cluster / np.maximum(energy, 1e-9), 0.0) > CALO_ENERGY_FRACTION_THRESHOLD

    return (
        np.asarray(pdg),
        np.asarray(genstatus),
        np.asarray(simstatus).astype(np.uint32),
        energy,
        eta,
        gp_in_tracker,
        gp_in_calo,
        np.asarray(px),
        np.asarray(py),
        np.asarray(pz),
        np.asarray(mass),
        gp_to_track,     # raw value, not just the boolean cut
        gp_to_cluster,    # raw value, not just the boolean cut
    )


all_pdg, all_status, all_sim, all_energy, all_eta = [], [], [], [], []
all_in_tracker, all_in_calo, all_gp_to_track, all_gp_to_cluster = [], [], [], []
for iev_ in range(ev.num_entries):
    pdg_, status_, sim_, energy_, eta_, in_trk_, in_calo_, *_rest, gtt_, gtc_ = analyze_event_visibility(iev_)
    all_pdg.append(pdg_)
    all_status.append(status_)
    all_sim.append(sim_)
    all_energy.append(energy_)
    all_eta.append(eta_)
    all_in_tracker.append(in_trk_)
    all_in_calo.append(in_calo_)
    all_gp_to_track.append(gtt_)
    all_gp_to_cluster.append(gtc_)

all_pdg = np.concatenate(all_pdg)
all_status = np.concatenate(all_status)
all_sim = np.concatenate(all_sim)
all_energy = np.concatenate(all_energy)
all_eta = np.concatenate(all_eta)
all_in_tracker = np.concatenate(all_in_tracker)
all_in_calo = np.concatenate(all_in_calo)
all_gp_to_track = np.concatenate(all_gp_to_track)
all_gp_to_cluster = np.concatenate(all_gp_to_cluster)

NEUTRINO_PDGS = (12, 14, 16)  # nu_e, nu_mu, nu_tau -- excluded from genjet in postprocessing.py,
                               # so they're excluded here too. Without this, neutrinos (which
                               # are correctly, physically invisible to tracker/calo) inflate the
                               # "dropped" and "mismatch" numbers below with energy that was never
                               # going to be reconstructable and isn't evidence of a real problem.
st1 = (all_status == 1) & ~np.isin(np.abs(all_pdg), NEUTRINO_PDGS)
n_st1 = st1.sum()

# the four categories are mutually exclusive and exhaustive by construction
track_only = st1 & all_in_tracker & ~all_in_calo
calo_only = st1 & ~all_in_tracker & all_in_calo
both = st1 & all_in_tracker & all_in_calo
neither = st1 & ~all_in_tracker & ~all_in_calo   # == dropped from targetjet

assert track_only.sum() + calo_only.sum() + both.sum() + neither.sum() == n_st1, \
    "the four categories should exactly partition status-1 particles"

n_neutrinos_excluded = ((all_status == 1) & np.isin(np.abs(all_pdg), NEUTRINO_PDGS)).sum()
print(f"excluded {n_neutrinos_excluded} status-1 neutrinos from this analysis "
      f"(matches postprocessing.py's own genjet definition)")


In [ ]:
# Readable summary as a DataFrame instead of a wall of prints.
total_e = all_energy[st1].sum()

summary = pandas.DataFrame([
    {"category": "track-only  (gp_in_tracker & ~gp_in_calo)", "n": track_only.sum(),
     "pct_of_n_st1": 100 * track_only.sum() / n_st1, "energy_GeV": all_energy[track_only].sum()},
    {"category": "calo-only   (gp_in_calo & ~gp_in_tracker)", "n": calo_only.sum(),
     "pct_of_n_st1": 100 * calo_only.sum() / n_st1, "energy_GeV": all_energy[calo_only].sum()},
    {"category": "both        (gp_in_tracker & gp_in_calo)", "n": both.sum(),
     "pct_of_n_st1": 100 * both.sum() / n_st1, "energy_GeV": all_energy[both].sum()},
    {"category": "neither -> DROPPED from targetjet", "n": neither.sum(),
     "pct_of_n_st1": 100 * neither.sum() / n_st1, "energy_GeV": all_energy[neither].sum()},
])
summary["pct_of_n_st1"] = summary["pct_of_n_st1"].round(1)
summary["energy_GeV"] = summary["energy_GeV"].round(1)
summary["pct_of_status1_energy"] = (100 * summary["energy_GeV"] / total_e).round(1)

print(f"status==1 particles across {ev.num_entries} events: {n_st1}  "
      f"(total status-1 energy: {total_e:.1f} GeV)")
summary
